# AutoResearch by Karpathy — Google Colab Setup

Autonomous LLM training experiments on a single NVIDIA GPU.

**Before running:** Go to `Runtime → Change runtime type → Hardware accelerator` and select **GPU** (T4, L4, A100, or H100).

> Default config (~45 GB VRAM) targets H100. This notebook auto-adjusts hyperparameters to match your Colab GPU.

## Step 1 — Check GPU

In [ ]:
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU detected. Please enable GPU: Runtime → Change runtime type → GPU')

gpu_info = result.stdout.strip()
print('GPU detected:', gpu_info)

# Parse VRAM (in MiB)
parts = [p.strip() for p in gpu_info.split(',')]
gpu_name = parts[0]
vram_mib = int(parts[1].replace(' MiB', ''))
compute_cap = parts[2]  # e.g. '7.5'
vram_gb = vram_mib / 1024

print(f'\nGPU Name  : {gpu_name}')
print(f'VRAM      : {vram_gb:.1f} GB')
print(f'Compute   : {compute_cap}')

# Tier classification
if vram_gb >= 70:
    GPU_TIER = 'H100'
elif vram_gb >= 38:
    GPU_TIER = 'A100'
elif vram_gb >= 20:
    GPU_TIER = 'L4'
else:
    GPU_TIER = 'T4'

print(f'\nAutoResearch tier: {GPU_TIER}')

## Step 2 — Clone Repository

In [ ]:
import os

REPO_URL = 'https://github.com/JanRuman/AutoResearch-Karpathy.git'
REPO_DIR = '/content/AutoResearch-Karpathy'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned, pulling latest changes...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

## Step 3 — Install `uv` and Dependencies

`uv` is the fast Python package manager used by this project. Dependencies include PyTorch 2.9.1 (CUDA 12.8) and Flash Attention 3 kernels.

In [ ]:
# Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh

# Add uv to PATH for this session
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ.get('PATH', '')

!uv --version

In [ ]:
# Sync all project dependencies (PyTorch CUDA 12.8 + Flash Attention 3)
# This may take 5-10 minutes on first run
!uv sync
print('\nDependencies installed successfully.')

## Step 4 — Configure Hyperparameters for Your GPU

The default config targets an H100 (~45 GB VRAM). The cell below patches `train.py` with safe defaults for your detected GPU tier.

| GPU Tier | VRAM | DEPTH | DEVICE_BATCH_SIZE | TOTAL_BATCH_SIZE | WINDOW_PATTERN | Est. VRAM |
|----------|------|-------|-------------------|-----------------|----------------|----------|
| H100     | 80 GB | 8    | 128               | 2^19 (~524K)    | SSSL           | ~45 GB   |
| A100     | 40 GB | 8    | 64                | 2^18 (~262K)    | SSSL           | ~24 GB   |
| L4       | 24 GB | 6    | 32                | 2^17 (~131K)    | SSSL           | ~14 GB   |
| T4       | 16 GB | 4    | 16                | 2^16 (~65K)     | L              | ~8 GB    |

In [ ]:
import re

# Hyperparameter profiles per GPU tier
GPU_PROFILES = {
    'H100': dict(
        DEPTH=8,
        DEVICE_BATCH_SIZE=128,
        TOTAL_BATCH_SIZE='2**19',
        WINDOW_PATTERN='SSSL',
        note='Default H100 config — no changes needed'
    ),
    'A100': dict(
        DEPTH=8,
        DEVICE_BATCH_SIZE=64,
        TOTAL_BATCH_SIZE='2**18',
        WINDOW_PATTERN='SSSL',
        note='A100 40GB config — reduced batch size'
    ),
    'L4': dict(
        DEPTH=6,
        DEVICE_BATCH_SIZE=32,
        TOTAL_BATCH_SIZE='2**17',
        WINDOW_PATTERN='SSSL',
        note='L4 24GB config — reduced depth & batch size'
    ),
    'T4': dict(
        DEPTH=4,
        DEVICE_BATCH_SIZE=16,
        TOTAL_BATCH_SIZE='2**16',
        WINDOW_PATTERN='L',
        note='T4 16GB config — small model, full-context attention only'
    ),
}

profile = GPU_PROFILES[GPU_TIER]
print(f'Applying profile for {GPU_TIER}: {profile["note"]}')

train_path = os.path.join(REPO_DIR, 'train.py')
with open(train_path, 'r') as f:
    src = f.read()

def replace_param(text, name, new_val):
    """Replace a hyperparameter assignment in the hyperparameters section."""
    # Match: NAME = <value>  # optional comment
    pattern = rf'^({re.escape(name)}\s*=\s*)([^\n#]+)(.*?)$'
    replacement = rf'\g<1>{new_val}\g<3>'
    new_text, n = re.subn(pattern, replacement, text, flags=re.MULTILINE)
    if n == 0:
        raise ValueError(f'Could not find parameter: {name}')
    return new_text

src = replace_param(src, 'DEPTH', str(profile['DEPTH']))
src = replace_param(src, 'DEVICE_BATCH_SIZE', str(profile['DEVICE_BATCH_SIZE']))
src = replace_param(src, 'TOTAL_BATCH_SIZE', profile['TOTAL_BATCH_SIZE'])
src = replace_param(src, 'WINDOW_PATTERN', f'"{profile["WINDOW_PATTERN"]}"')

with open(train_path, 'w') as f:
    f.write(src)

print(f'  DEPTH              = {profile["DEPTH"]}')
print(f'  DEVICE_BATCH_SIZE  = {profile["DEVICE_BATCH_SIZE"]}')
print(f'  TOTAL_BATCH_SIZE   = {profile["TOTAL_BATCH_SIZE"]}')
print(f'  WINDOW_PATTERN     = {profile["WINDOW_PATTERN"]}')
print('\ntrain.py patched successfully.')

## Step 5 — Download Data and Train Tokenizer

This is a **one-time setup** (~2 minutes). Downloads training data shards from Hugging Face into `~/.cache/autoresearch/` and trains an 8,192-token BPE tokenizer.

> **Note:** Colab `/content` is ephemeral — data is lost when the session ends. Re-run this cell when you start a new session, or mount Google Drive first (see optional cell below).

In [ ]:
# OPTIONAL: Mount Google Drive to persist data between Colab sessions.
# If you mount Drive, the tokenizer and dataset cache will survive reboots.
# Skip this cell if you prefer to re-download each session.

MOUNT_DRIVE = False  # Set to True to enable

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Redirect cache to Drive
    cache_dir = '/content/drive/MyDrive/autoresearch_cache'
    os.makedirs(cache_dir, exist_ok=True)
    os.environ['HOME'] = '/content'  # keep home, symlink cache
    cache_target = os.path.expanduser('~/.cache/autoresearch')
    if not os.path.islink(cache_target):
        os.makedirs(os.path.dirname(cache_target), exist_ok=True)
        if os.path.exists(cache_target):
            import shutil
            shutil.copytree(cache_target, cache_dir, dirs_exist_ok=True)
            shutil.rmtree(cache_target)
        os.symlink(cache_dir, cache_target)
    print(f'Cache linked to Google Drive: {cache_dir}')
else:
    print('Using ephemeral /root/.cache (data lost on session end)')

In [ ]:
# Check if data already exists
cache_path = os.path.expanduser('~/.cache/autoresearch')
data_path = os.path.join(cache_path, 'data')
tok_path  = os.path.join(cache_path, 'tokenizer')

has_data = os.path.exists(data_path) and len(os.listdir(data_path)) > 0
has_tok  = os.path.exists(tok_path)  and len(os.listdir(tok_path))  > 0

if has_data and has_tok:
    print('Data and tokenizer already exist — skipping prepare.py')
    print(f'  Data shards : {len(os.listdir(data_path))}')
    print(f'  Tokenizer   : {os.listdir(tok_path)}')
else:
    print('Running prepare.py (one-time setup, ~2 min)...')
    !uv run prepare.py

## Step 6 — Run a Single Baseline Experiment (~5 min)

This verifies the full pipeline. Training runs for exactly **5 minutes** (wall clock, excluding startup).
Key output metric: `val_bpb` (validation bits per byte — lower is better).

In [ ]:
# Run baseline training (~5 min + ~30s startup)
!uv run train.py 2>&1 | tee run.log
print('\n--- Key Metrics ---')
!grep -E '^val_bpb:|^peak_vram_mb:|^num_params_M:|^mfu_percent:' run.log

## Step 7 — Set Up Experiment Branch and results.tsv

For autonomous research, each session should use a dedicated git branch and a `results.tsv` tracking file.

In [ ]:
from datetime import datetime

# Auto-generate a tag like mar18 or override manually
RUN_TAG = datetime.now().strftime('%b%d').lower()  # e.g. 'mar18'
BRANCH_NAME = f'autoresearch/{RUN_TAG}'

print(f'Experiment branch: {BRANCH_NAME}')

# Create branch if it doesn't exist
!git checkout -b {BRANCH_NAME} 2>/dev/null || git checkout {BRANCH_NAME}

# Commit the GPU-tuned train.py as the baseline
!git add train.py
!git commit -m "baseline: {GPU_TIER} config (depth={profile['DEPTH']}, batch={profile['TOTAL_BATCH_SIZE']})" || echo 'Nothing to commit'

# Initialize results.tsv if not present
results_path = os.path.join(REPO_DIR, 'results.tsv')
if not os.path.exists(results_path):
    with open(results_path, 'w') as f:
        f.write('commit\tval_bpb\tmemory_gb\tstatus\tdescription\n')
    print('Created results.tsv')
else:
    print('results.tsv already exists')

print('\nSetup complete. Ready for autonomous research loop.')

## Step 8 — Launch Autonomous Research Agent

Paste the prompt below into Claude Code (or any supported agent) to start the autonomous experiment loop.
The agent will read `program.md`, modify `train.py`, run 5-minute experiments, and iterate — tracking results in `results.tsv`.

**Tip:** The loop runs indefinitely. Stop it manually when you're done (or let it run overnight).

In [ ]:
prompt = f"""Hi! Have a look at program.md and let's kick off a new experiment.

Setup is already done:
- Branch: {BRANCH_NAME}
- GPU: {GPU_TIER} ({vram_gb:.0f} GB VRAM)
- Data and tokenizer are in ~/.cache/autoresearch/
- results.tsv is initialized
- train.py is patched for this GPU (DEPTH={profile['DEPTH']}, DEVICE_BATCH_SIZE={profile['DEVICE_BATCH_SIZE']})

The baseline val_bpb from Step 6 is already logged.
Please start the experiment loop — run `uv run train.py > run.log 2>&1` for each experiment.
Never stop until I manually interrupt you.
"""

print('=== Copy this prompt into your AI agent ===')
print(prompt)

## Step 9 — Analyze Results

Run this cell at any time to visualize experiment progress from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

results_path = os.path.join(REPO_DIR, 'results.tsv')
df = pd.read_csv(results_path, sep='\t')

if df.empty:
    print('No results yet. Run Step 6 first to get the baseline.')
else:
    print(f'Total experiments: {len(df)}')
    print(f'Kept             : {(df.status == "keep").sum()}')
    print(f'Discarded        : {(df.status == "discard").sum()}')
    print(f'Crashed          : {(df.status == "crash").sum()}')
    print()

    valid = df[df.val_bpb > 0].copy()
    valid['idx'] = range(len(valid))
    kept = valid[valid.status == 'keep']

    # Running best
    valid['running_best'] = valid['val_bpb'].cummin()

    fig, ax = plt.subplots(figsize=(12, 5))

    color_map = {'keep': 'green', 'discard': 'salmon', 'crash': 'gray'}
    for _, row in valid.iterrows():
        ax.scatter(row['idx'], row['val_bpb'],
                   color=color_map.get(row['status'], 'blue'), s=40, zorder=3)

    ax.plot(valid['idx'], valid['running_best'], 'b-', linewidth=2, label='Running best', zorder=2)

    ax.set_xlabel('Experiment #')
    ax.set_ylabel('val_bpb (lower is better)')
    ax.set_title(f'AutoResearch Progress — {GPU_TIER} GPU  |  {len(df)} experiments')

    patches = [mpatches.Patch(color=v, label=k) for k, v in color_map.items()]
    patches.append(plt.Line2D([0],[0], color='b', linewidth=2, label='Running best'))
    ax.legend(handles=patches)

    plt.tight_layout()
    plt.savefig(os.path.join(REPO_DIR, 'progress.png'), dpi=150)
    plt.show()

    if not kept.empty:
        best = kept.loc[kept.val_bpb.idxmin()]
        print(f'Best val_bpb: {best.val_bpb:.6f} (commit {best.commit}) — {best.description}')

    print('\nTop 5 kept experiments:')
    print(kept.nsmallest(5, 'val_bpb')[['commit','val_bpb','memory_gb','description']].to_string(index=False))

## Tips & Troubleshooting

### OOM (Out of Memory)
If you see a CUDA OOM error, reduce `DEVICE_BATCH_SIZE` in `train.py` (halve it, e.g. 16 → 8) and retry.

### T4 / Low-VRAM GPUs
The README recommends using a smaller dataset for very small GPUs. Consider setting `WINDOW_PATTERN = "L"` (full context, no sliding window complexity) in `train.py`.

### Slow Downloads
`prepare.py` downloads data shards from Hugging Face. Colab has good bandwidth so this is usually fast. If it's slow, you can limit the number of shards by editing `prepare.py` (not recommended for production runs).

### Session Disconnect
Colab sessions disconnect after ~90 min of inactivity or 12 hours max. Use **Google Drive mounting** (Step 5 optional cell) to persist the data cache, and commit+push results frequently.

### Flash Attention 3 on T4
The code auto-selects the Flash Attention 3 kernel variant:
- H100 (compute 9.0): `varunneal/flash-attention-3` (FA3 native)
- Other GPUs (T4, L4, A100): `kernels-community/flash-attn3` (compatible fallback)

If kernel loading fails, check the `kernels` package version: `uv run pip show kernels`.